# 03 - Embeddings e indexacion en Qdrant

Objetivo: unir documentos, chunking, embeddings de Ollama e indexacion en Qdrant.


In [ ]:
import os
import uuid
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

load_dotenv(Path("..").resolve() / ".env")

OLLAMA_PORT = os.getenv("OLLAMA_PORT", "11434")
QDRANT_PORT = os.getenv("QDRANT_PORT", "6333")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "embeddinggemma")
COLLECTION_NAME = os.getenv("COLLECTION_NAME", "frasohome_docs")
CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "650"))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "100"))

DOCS_DIR = Path("..").resolve() / "docs"
OLLAMA_BASE_URL = f"http://127.0.0.1:{OLLAMA_PORT}"
QDRANT_URL = f"http://127.0.0.1:{QDRANT_PORT}"


In [ ]:
loader = DirectoryLoader(
    str(DOCS_DIR),
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(documents)

for idx, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = idx

print(f"Documentos: {len(documents)}")
print(f"Chunks: {len(chunks)}")


In [ ]:
embedding_model = OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=OLLAMA_BASE_URL)
texts = [chunk.page_content for chunk in chunks]
vectors = embedding_model.embed_documents(texts)
vector_size = len(vectors[0])

client = QdrantClient(url=QDRANT_URL)
client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
)

points = []
for chunk, vector in zip(chunks, vectors):
    points.append(
        PointStruct(
            id=str(uuid.uuid4()),
            vector=vector,
            payload={
                "source": chunk.metadata.get("source", "desconocido"),
                "chunk_id": chunk.metadata["chunk_id"],
                "text": chunk.page_content,
            },
        )
    )

client.upsert(collection_name=COLLECTION_NAME, points=points)
client.get_collection(COLLECTION_NAME)


In [ ]:
question = "Que debo hacer si faltan tornillos en la caja?"
query_vector = embedding_model.embed_query(question)
hits = client.search(collection_name=COLLECTION_NAME, query_vector=query_vector, limit=3)

for hit in hits:
    print("=" * 80)
    print({
        "score": round(hit.score, 4),
        "source": hit.payload.get("source"),
        "chunk_id": hit.payload.get("chunk_id"),
    })
    print(hit.payload.get("text", "")[:500])
